# **КЛАССИФИКАЦИЯ ДАННЫХ И ПЕРВИЧНАЯ ОЦЕНКА КАЧЕСТВА ДАННЫХ**
---

Погода: измерения (date, temp, humidity, wind).

Обязательные требования к индивидуальному набору данных:

* не менее 2 пропусков (NaN/None);
* не менее 1 дубликата строки или дубликата по ключу;
* не менее 2 примеров шума (аномальные значения, некорректные форматы, лишние пробелы и т.п.).
---

## **1. Подключение библиотек**

In [2]:
import pandas as pd
import numpy as np
import json

## **2. Подготовка структурированных данных**

In [3]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_colwidth', None)

In [4]:
df_struct = pd.DataFrame({
    'date': ['16-01-2026', '17-01-2026', '18-01-2026', '19-01-2026', '20-01-2026', '20-01-2026'],
    'temp': [-15, -16, 19, None, -17, -17],
    'humidity': [4, 4, 2, 120, 2, 2],
    'wind': [None, 12, 10, 5, 8, 8]
})
df_struct

,date,temp,humidity,wind
0,16-01-2026,-15.0,4,NaN
1,17-01-2026,-16.0,4,12.0
2,18-01-2026,19.0,2,10.0
3,19-01-2026,NaN,120,5.0
4,20-01-2026,-17.0,2,8.0
5,20-01-2026,-17.0,2,8.0


## **3. Классификация данных**

In [5]:
types_table = pd.DataFrame({
    'Набор данных': ['df_struct'],
    'Тип данных': ['структурированные'],
    'Почему': [
        'таблица: строки и столбцы, фиксированная схема',
    ]
})
types_table

,Набор данных,Тип данных,Почему
0,df_struct,структурированные,"таблица: строки и столбцы, фиксированная схема"


## **4. Идентификация пропусков**

In [6]:
missing_count = df_struct.isna().sum()
missing_percent = df_struct.isna().mean() * 100

pd.DataFrame({
    'missing_count': missing_count,
    'missing_percent': missing_percent.round(1)
})

,missing_count,missing_percent
date,0,0.0
temp,1,16.7
humidity,0,0.0
wind,1,16.7


## **5. Идентификация дубликатов**

In [7]:
df_struct.duplicated().sum()

1

In [8]:
df_struct.duplicated(subset=['date']).sum()
df_struct[df_struct.duplicated(subset=['date'], keep=False)]

,date,temp,humidity,wind
4,20-01-2026,-17.0,2,8.0
5,20-01-2026,-17.0,2,8.0


## **6. Идентификация шума**

In [9]:
df_struct[['temp', 'humidity', 'wind']].describe()

,temp,humidity,wind
count,5.00000,6.000000,5.000000
mean,-9.20000,22.333333,8.600000
std,15.78607,47.856731,2.607681
min,-17.00000,2.000000,5.000000
25%,-17.00000,2.000000,8.000000
50%,-16.00000,3.000000,8.000000
75%,-15.00000,4.000000,10.000000
max,19.00000,120.000000,12.000000


In [10]:
# проверка по правилам
noise_temp = df_struct[(df_struct['temp'].notna()) & ((df_struct['temp'] >= 0 | (df_struct['temp'] <= -50)))]
noise_wind = df_struct[df_struct['wind'].notna() & ((df_struct['wind'] >= 30) | (df_struct['wind'] < 0))]
noise_humidity = df_struct[df_struct['humidity'].notna() & ((df_struct['humidity'] < 0) | (df_struct['humidity'] > 100))]
noise_wind_nan = df_struct[df_struct['wind'].isna()]
noise_temp_nan = df_struct[df_struct['temp'].isna()]
noise_humidity_nan = df_struct[df_struct['humidity'].isna()]

In [13]:
noise_temp

,date,temp,humidity,wind
2,18-01-2026,19.0,2,10.0


In [14]:
noise_wind

,date,temp,humidity,wind


In [15]:
noise_humidity

,date,temp,humidity,wind
3,19-01-2026,NaN,120,5.0


In [16]:
noise_wind_nan

,date,temp,humidity,wind
0,16-01-2026,-15.0,4,NaN


In [17]:
noise_temp_nan

,date,temp,humidity,wind
3,19-01-2026,NaN,120,5.0


In [18]:
noise_humidity_nan

,date,temp,humidity,wind


## **7. Чек-лист качества данных**

In [19]:
total_missing = int(df_struct.isna().sum().sum())
dup_rows = int(df_struct.duplicated().sum())
dup_by_id = int(df_struct.duplicated(subset=['date']).sum())
noise_temp_count = len(noise_temp)
noise_wind_count = len(noise_wind)
noise_humidity_count = len(noise_humidity)

In [21]:
checklist = pd.DataFrame({
    'Критерий': ['Полнота', 'Уникальность', 'Корректность значений'],
    'Проверка': [
        'Пропуски по всем столбцам',
        'Дубликаты строк и дубликаты по ключу id',
        'Аномалии temp/wind/humidity + шум в date'
    ],
    'Метрика': [
        f'Пропуски (всего): {total_missing}',
        f'Дубликаты строк: {dup_rows}; по date: {dup_by_id}',
        f'temp аном:{noise_temp_count}; wind аном:{noise_wind_count}; humidity аном:{noise_humidity_count}'
    ],
    'Статус': [
        'Проблема' if total_missing > 0 else 'OK',
        'Проблема' if (dup_rows > 0 or dup_by_id > 0) else 'OK',
        'Проблема' if (noise_temp_count > 0 or noise_wind_count > 0 or noise_humidity_count > 0) else 'OK'
    ],
    'Комментарий': [
        'Требуется обработка пропусков',
        'Требуется проверка/удаление дублей',
        'Требуется фильтрация/исправление значений'
    ]
})
checklist

,Критерий,Проверка,Метрика,Статус,Комментарий
0,Полнота,Пропуски по всем столбцам,Пропуски (всего): 2,Проблема,Требуется обработка пропусков
1,Уникальность,Дубликаты строк и дубликаты по ключу id,Дубликаты строк: 1; по date: 1,Проблема,Требуется проверка/удаление дублей
2,Корректность значений,Аномалии temp/wind/humidity + шум в date,temp аном:1; wind аном:0; humidity аном:1,Проблема,Требуется фильтрация/исправление значений


## Контрольные вопросы

**1. Чем структурированные данные отличаются от полуструктурированных и неструктурированных?**

Структурированные данные хранятся в таблицах, имеют четко определенные типы (числы, даты, категории).
Полуструктурированные данные имеют некоторую организацию, но не определенную схему. Например, JSON, XML, логи событий.
Неструктурированные данные не имеют зараннее заданной структурированный формы. Например, текстовые документы, изображения, аудио, видео.

**2. Что такое пропуски в данных и почему они важны для анализа?**

Пропуски - это отсутствующие значения (NaN, пустые ячейки).
Они важны, поскольку они искажают статистику, алгоритмы ML также часто не умеют с ними работать.

**3. В чём опасность дубликатов для агрегирования и расчёта показателей?**

Дубликаты - повторяющиеся строки или записи.
Они искажают агрегаты (суммы, средние, количество). А также могут привести к неверным бизнес-решениям.

**4. Что понимается под шумом в данных? Приведите примеры.**

Шум - это случайные или нерелевантные значения, не отражающие закономерность.
Например, ошибки ввода (возраст - 999 лет).

**5. Зачем формируется чек-лист качества данных перед аналитической обработкой?**

Чек-лист формируется для системной проверки перед анализом.
Он необходим для обеспечения надежности анализма и моделей, исключения искажений и ошибок.